# Notebook 2: The Cost of Attention

**ML4FP --- Machine Learning for Fundamental Physics**

---

In Notebook 1, we built self-attention where every particle interacts with every other particle. This requires computing an $n \times n$ attention matrix, where $n$ is the number of particles:
Doubling $n$ makes the cost 4 times larger -- quadratic scaling.

This notebook covers:
1. Measuring the quadratic cost empirically
2. Why most of the attention matrix carries little information
3. **Linformer**, which compresses attention to linear cost
4. Side-by-side training comparison on jet data

---
## 1. Setup

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (8, 5),
    'font.size': 13,
    'axes.labelsize': 14,
    'axes.titlesize': 15,
    'legend.fontsize': 11,
    'figure.dpi': 100,
})

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

---
## 2. Measuring the Cost of Attention

We time a simple attention function for increasing sequence lengths. If attention is $O(n^2)$, doubling $n$ should roughly quadruple the runtime.

In [ ]:
def standard_attention(Q, K, V):
    """Standard scaled dot-product attention.
    
    Args:
        Q, K, V: shape (batch, heads, n, d_k)
    Returns:
        output: shape (batch, heads, n, d_k)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)  # (B, H, n, n)
    attn = F.softmax(scores, dim=-1)                                # (B, H, n, n)
    return torch.matmul(attn, V)                                    # (B, H, n, d_k)

In [ ]:
# Time attention for increasing sequence lengths
seq_lengths = [32, 64, 128, 256, 512, 1024, 2048]
n_heads = 2
d_k = 16
n_repeats = 5

times = []

print(f"{'n':>6s}  {'Time (ms)':>10s}  {'vs previous':>12s}")
print("-" * 34)

for i, n in enumerate(seq_lengths):
    Q = torch.randn(1, n_heads, n, d_k)
    K = torch.randn(1, n_heads, n, d_k)
    V = torch.randn(1, n_heads, n, d_k)
    
    # Warm-up
    _ = standard_attention(Q, K, V)
    
    # Time it
    start = time.time()
    for _ in range(n_repeats):
        _ = standard_attention(Q, K, V)
    elapsed = (time.time() - start) / n_repeats * 1000  # ms
    times.append(elapsed)
    
    ratio = f"{elapsed / times[i-1]:.1f}x" if i > 0 else "---"
    print(f"{n:>6d}  {elapsed:>10.2f}  {ratio:>12s}")

times = np.array(times)
seq_lengths_arr = np.array(seq_lengths, dtype=float)

i128 = seq_lengths.index(128)
i2048 = seq_lengths.index(2048)
growth = times[i2048] / times[i128]
print("\nFrom n=128 to n=2048, the sequence length grows by 16x while runtime grows")
print(f"from about {times[i128]:.2f} ms to {times[i2048]:.2f} ms, i.e. by {growth:.1f}x")
print("in this CPU benchmark. Pairwise attention stays cheap for jets, but it gets")
print("expensive quickly once the object has hundreds or thousands of constituents.")

In [ ]:
# Plot: time vs sequence length
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(seq_lengths, times, 'o-', color='#e74c3c', markersize=8, linewidth=2, label='Measured')
# Fit a * n^2
a_fit = np.sum(times * seq_lengths_arr**2) / np.sum(seq_lengths_arr**4)
n_fit = np.linspace(seq_lengths[0], seq_lengths[-1], 200)
ax.plot(n_fit, a_fit * n_fit**2, '--', color='gray', linewidth=1.5, label=r'Fit: $t \propto n^2$')
ax.set_xlabel('Sequence length $n$')
ax.set_ylabel('Time (ms)')
ax.set_title('Attention time grows quadratically')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Linformer

### Intuition

The $O(n^2)$ cost comes from forming the full $n \times n$ score matrix. **Linformer** (Wang et al., 2020) avoids it by compressing the $n$ keys and values down to $k$ summary vectors with learned projection matrices, so each particle attends to $k$ summaries instead of $n$ individual particles. With $k$ fixed, the cost grows linearly in $n$.

### The math

**Standard attention** computes an $n \times n$ matrix:

$$\text{Standard:} \quad \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V \quad \text{where } QK^T \in \mathbb{R}^{n \times n}$$

**Linformer** projects $K$ and $V$ from length $n$ down to length $k$:

$$\tilde{K} = E \cdot K, \quad \tilde{V} = F \cdot V \qquad \text{where } E, F \in \mathbb{R}^{k \times n}$$

$$\text{Linformer:} \quad \text{softmax}\!\left(\frac{Q \tilde{K}^T}{\sqrt{d_k}}\right) \tilde{V} \quad \text{where } Q\tilde{K}^T \in \mathbb{R}^{n \times k}$$

$E$ and $F$ are learned projection matrices.

### Cost comparison

| | Standard | Linformer (k=32, n=128) |
|---|---|---|
| Attention matrix size | $n \times n = 16{,}384$ | $n \times k = 4{,}096$ |
| Complexity | $O(n^2)$ | $O(nk)$ |
| Speedup | --- | $n/k = 4\times$ |

Since $k$ is fixed, the cost scales linearly in $n$.

---
## 4. Implementing Linformer Attention

Standard attention:

```python
scores = Q @ K.T / sqrt(d_k)     # n x n
attn = softmax(scores)
output = attn @ V
```

Linformer adds two projection lines before computing scores:

```python
K_proj = E @ K          # n x d_k  -->  k x d_k
V_proj = F @ V          # n x d_k  -->  k x d_k
scores = Q @ K_proj.T   # n x k
attn = softmax(scores)
output = attn @ V_proj
```

### Exercise: Fill in the Linformer projection

Complete the `forward` method below -- 3 lines.

In [ ]:
class LinformerAttentionExercise(nn.Module):
    """Linformer multi-head self-attention."""
    
    def __init__(self, d_model, n_heads, seq_len, k=32, share_kv=True):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.k = k
        
        # Standard Q, K, V projections
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        # Linformer projection: compress sequence from n -> k
        self.E = nn.Parameter(torch.randn(k, seq_len) / seq_len**0.5)
        if share_kv:
            self.F = self.E  # reuse same projection for keys and values
        else:
            self.F = nn.Parameter(torch.randn(k, seq_len) / seq_len**0.5)
    
    def forward(self, x, mask=None):
        B, n, d = x.shape
        h, d_k = self.n_heads, self.d_k
        
        # Compute Q, K, V and reshape to (B, h, n, d_k)
        Q = self.W_Q(x).view(B, n, h, d_k).transpose(1, 2)
        K = self.W_K(x).view(B, n, h, d_k).transpose(1, 2)
        V = self.W_V(x).view(B, n, h, d_k).transpose(1, 2)
        
        # === YOUR CODE: fill in the 3 lines below ===
        # Line 1: Project K from (B, h, n, d_k) to (B, h, k, d_k) using self.E
        K_proj = ...
        # Line 2: Project V the same way using self.F
        V_proj = ...
        # Line 3: Compute attention scores between Q and K_proj (don't forget to scale!)
        scores = ...
        # === END YOUR CODE ===
        
        attn = F.softmax(scores, dim=-1)          # (B, h, n, k)
        out = torch.matmul(attn, V_proj)           # (B, h, n, d_k)
        
        # Concatenate heads and apply output projection
        out = out.transpose(1, 2).contiguous().view(B, n, h * d_k)
        return self.W_O(out)

<details>
<summary><b>Click for solution</b></summary>

The completed solution is in the minimized code cell below.

</details>

In [ ]:
class LinformerAttention(nn.Module):
    """Linformer multi-head self-attention."""
    
    def __init__(self, d_model, n_heads, seq_len, k=32, share_kv=True):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.k = k
        
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        
        self.E = nn.Parameter(torch.randn(k, seq_len) / seq_len**0.5)
        if share_kv:
            self.F = self.E
        else:
            self.F = nn.Parameter(torch.randn(k, seq_len) / seq_len**0.5)
    
    def forward(self, x, mask=None):
        B, n, d = x.shape
        h, d_k = self.n_heads, self.d_k
        
        Q = self.W_Q(x).view(B, n, h, d_k).transpose(1, 2)  # (B, h, n, d_k)
        K = self.W_K(x).view(B, n, h, d_k).transpose(1, 2)
        V = self.W_V(x).view(B, n, h, d_k).transpose(1, 2)
        
        # Apply mask before projection (zero out padding)
        if mask is not None:
            mask_exp = mask.unsqueeze(1).unsqueeze(-1).float()  # (B, 1, n, 1)
            K = K * mask_exp
            V = V * mask_exp
        
        E = self.E[:, :n]   # handle shorter sequences
        F_mat = self.F[:, :n]
        
        # The Linformer trick: project K and V from n -> k
        K_proj = torch.matmul(E, K)      # (k, n) @ (B, h, n, d_k) -> (B, h, k, d_k)
        V_proj = torch.matmul(F_mat, V)  # (k, n) @ (B, h, n, d_k) -> (B, h, k, d_k)
        
        # Attention with projected keys: n x k instead of n x n
        scores = torch.matmul(Q, K_proj.transpose(-2, -1)) / (d_k ** 0.5)  # (B, h, n, k)
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V_proj)  # (B, h, n, d_k)
        
        out = out.transpose(1, 2).contiguous().view(B, n, h * d_k)
        return self.W_O(out)

In [ ]:
# Test: verify shapes
d_model_test = 32
n_heads_test = 2
seq_len_test = 128
k_test = 32

attn_module = LinformerAttention(d_model_test, n_heads_test, seq_len_test, k=k_test)
x_test = torch.randn(4, seq_len_test, d_model_test)
out_test = attn_module(x_test)

print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out_test.shape}")
assert out_test.shape == x_test.shape, "Shapes don't match!"
print("Shape check passed!")

In [ ]:
# Benchmark: Standard vs Linformer

def linformer_attention_fn(Q, K, V, E, F_mat):
    """Functional Linformer attention for benchmarking."""
    d_k = Q.size(-1)
    K_proj = torch.matmul(E, K)
    V_proj = torch.matmul(F_mat, V)
    scores = torch.matmul(Q, K_proj.transpose(-2, -1)) / (d_k ** 0.5)
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V_proj)


bench_lengths = [32, 64, 128, 256, 512, 1024, 2048]
k_bench = 32
n_repeats = 5

std_times = []
lin_times = []

print(f"{'n':>6s}  {'Standard (ms)':>14s}  {'Linformer (ms)':>15s}  {'Speedup':>8s}")
print("-" * 50)

for n in bench_lengths:
    Q = torch.randn(1, 2, n, 16)
    K = torch.randn(1, 2, n, 16)
    V = torch.randn(1, 2, n, 16)
    E = torch.randn(k_bench, n) / n**0.5
    F_mat = torch.randn(k_bench, n) / n**0.5
    
    # Warm-up
    _ = standard_attention(Q, K, V)
    _ = linformer_attention_fn(Q, K, V, E, F_mat)
    
    start = time.time()
    for _ in range(n_repeats):
        _ = standard_attention(Q, K, V)
    t_std = (time.time() - start) / n_repeats * 1000
    std_times.append(t_std)
    
    start = time.time()
    for _ in range(n_repeats):
        _ = linformer_attention_fn(Q, K, V, E, F_mat)
    t_lin = (time.time() - start) / n_repeats * 1000
    lin_times.append(t_lin)
    
    speedup = t_std / t_lin if t_lin > 0 else float('inf')
    print(f"{n:>6d}  {t_std:>14.2f}  {t_lin:>15.2f}  {speedup:>7.1f}x")

std_times = np.array(std_times)
lin_times = np.array(lin_times)

i128 = bench_lengths.index(128)
i2048 = bench_lengths.index(2048)
speedup_2048 = std_times[i2048] / lin_times[i2048]
print("\nAt n=128, both methods are already cheap on CPU, so there is little to gain.")
print(f"At n=2048 in this benchmark, Linformer is {speedup_2048:.1f}x faster because")
print("it never forms the full n x n attention matrix.")

In [ ]:
# Plot both on the same graph
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(bench_lengths, std_times, 'o-', color='#e74c3c', markersize=7, linewidth=2,
        label='Standard $O(n^2)$')
ax.plot(bench_lengths, lin_times, 's-', color='#2ecc71', markersize=7, linewidth=2,
        label=f'Linformer $O(nk)$, k={k_bench}')
ax.axvline(x=128, color='#3498db', linestyle='--', alpha=0.5, label='n=128 (our jets)')
ax.set_xlabel('Sequence length $n$')
ax.set_ylabel('Time (ms)')
ax.set_title('Standard vs Linformer: Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("For small n (like 128), standard attention is fine.")
print("But as n grows, Linformer's advantage becomes clear!")

---
## 5. Training Comparison on JetClass

Faster attention is only useful if the model still performs. We train two tiny jet classifiers -- one with standard attention, one with Linformer -- and compare accuracy.

To keep the comparison fair, we reuse the **exact same** small binary QCD-vs-top task from Notebook 1. The only change is the attention mechanism.

Setup:
- `d_model=32`, `n_heads=2`, `d_ff=64`, `n_layers=2`
- 6,000 binary-task jets (4,800 train, 1,200 test)
- `batch_size=256`, 5 epochs

In [ ]:
# Download dataset if needed
from tutorial_data import ensure_jetclass_example

data_path = ensure_jetclass_example('../data')

In [ ]:
from dataloader import read_file
from tutorial_data import prepare_small_binary_task

x_particles, x_jets, y = read_file(data_path)
task = prepare_small_binary_task(x_particles, y, n_use=6000, seed=42)

binary_classes = task['binary_classes']
binary_names = task['binary_names']
x_train = task['x_train']
mask_train = task['mask_train']
y_train = task['y_train']
x_test = task['x_test']
mask_test = task['mask_test']
y_test = task['y_test']

print(f"Loaded {x_particles.shape[0]} jets from JetClass")
print(f"Shared task: {binary_names[0]} (class {binary_classes[0]}) vs {binary_names[1]} (class {binary_classes[1]})")
print(f"Train: {len(x_train)} jets   Test: {len(x_test)} jets")
print(f"Average particles per jet: {mask_train.float().sum(dim=1).mean():.0f}")

In [ ]:
# Reuse the same train/test split as Notebook 1
n_train = len(x_train)
n_val = len(x_test)

train_dataset = TensorDataset(x_train, mask_train, y_train)
val_dataset = TensorDataset(x_test, mask_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

print(f"Training: {n_train} jets, Held-out: {n_val} jets")

In [ ]:
# Build a simple jet classifier that can swap attention types

class StandardMultiHeadAttention(nn.Module):
    """Standard multi-head self-attention."""
    def __init__(self, d_model, n_heads, **kwargs):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        B, n, d = x.shape
        h, d_k = self.n_heads, self.d_k
        Q = self.W_Q(x).view(B, n, h, d_k).transpose(1, 2)
        K = self.W_K(x).view(B, n, h, d_k).transpose(1, 2)
        V = self.W_V(x).view(B, n, h, d_k).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        out = torch.matmul(attn, V)
        out = out.transpose(1, 2).contiguous().view(B, n, h * d_k)
        return self.W_O(out)


class TransformerBlock(nn.Module):
    """One transformer layer: attention + feed-forward."""
    def __init__(self, d_model, n_heads, d_ff=64,
                 attention_class=StandardMultiHeadAttention, dropout=0.1, **attn_kwargs):
        super().__init__()
        self.attention = attention_class(d_model, n_heads, **attn_kwargs)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        x = x + self.dropout(self.attention(self.norm1(x), mask=mask))
        x = x + self.ffn(self.norm2(x))
        return x


class JetClassifier(nn.Module):
    """Same tiny classifier as Notebook 1, with swappable attention."""
    def __init__(self, input_dim, d_model, n_heads, d_ff, n_layers, n_classes,
                 attention_class=StandardMultiHeadAttention, dropout=0.1, **attn_kwargs):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff=d_ff, attention_class=attention_class,
                            dropout=dropout, **attn_kwargs)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )
    
    def forward(self, x, mask=None):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x, mask=mask)
        x = self.norm(x)
        # Average pooling over non-masked particles
        if mask is not None:
            mask_f = mask.unsqueeze(-1).float()
            x = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)
        return self.classifier(x)


print("Model classes defined.")

In [ ]:
# Training function

def train_model(model, train_loader, val_loader, n_epochs=5, lr=1e-3):
    """Train a model and return training history."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(n_epochs):
        # Train
        model.train()
        total_loss, correct, total = 0, 0, 0
        for X_b, mask_b, Y_b in train_loader:
            X_b = X_b.to(device)
            mask_b = mask_b.to(device)
            Y_b = Y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b, mask=mask_b)
            loss = criterion(logits, Y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_b.size(0)
            correct += (logits.argmax(1) == Y_b).sum().item()
            total += X_b.size(0)
        history['train_loss'].append(total_loss / total)
        history['train_acc'].append(correct / total)
        
        # Validate
        model.eval()
        total_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for X_b, mask_b, Y_b in val_loader:
                X_b = X_b.to(device)
                mask_b = mask_b.to(device)
                Y_b = Y_b.to(device)
                logits = model(X_b, mask=mask_b)
                loss = criterion(logits, Y_b)
                total_loss += loss.item() * X_b.size(0)
                correct += (logits.argmax(1) == Y_b).sum().item()
                total += X_b.size(0)
        history['val_loss'].append(total_loss / total)
        history['val_acc'].append(correct / total)
        
        print(f"  Epoch {epoch+1}/{n_epochs}  "
              f"train_acc={history['train_acc'][-1]:.3f}  "
              f"val_acc={history['val_acc'][-1]:.3f}")
    
    return history

print("Training function defined.")

In [ ]:
# Hyperparameters (tiny model for fast training)
input_dim = 4
d_model = 32
n_heads = 2
d_ff = 64
n_layers = 2
n_classes = 2
n_epochs = 5

# --- Train standard transformer ---
print("=" * 50)
print("Training STANDARD Transformer")
print("=" * 50)

model_std = JetClassifier(
    input_dim=input_dim, d_model=d_model, n_heads=n_heads,
    d_ff=d_ff, n_layers=n_layers, n_classes=n_classes,
    attention_class=StandardMultiHeadAttention,
)
print(f"Parameters: {sum(p.numel() for p in model_std.parameters()):,}")
history_std = train_model(model_std, train_loader, val_loader, n_epochs=n_epochs)

In [ ]:
# --- Train Linformer transformer ---
print("=" * 50)
print("Training LINFORMER Transformer (k=32)")
print("=" * 50)

model_lin = JetClassifier(
    input_dim=input_dim, d_model=d_model, n_heads=n_heads,
    d_ff=d_ff, n_layers=n_layers, n_classes=n_classes,
    attention_class=LinformerAttention,
    seq_len=128, k=32, share_kv=True,
)
print(f"Parameters: {sum(p.numel() for p in model_lin.parameters()):,}")
history_lin = train_model(model_lin, train_loader, val_loader, n_epochs=n_epochs)

In [ ]:
# Plot training curves side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs = range(1, n_epochs + 1)

# Loss
ax = axes[0]
ax.plot(epochs, history_std['train_loss'], 'o--', color='#e74c3c', alpha=0.5, label='Standard (train)')
ax.plot(epochs, history_std['val_loss'], 'o-', color='#e74c3c', linewidth=2, label='Standard (held-out)')
ax.plot(epochs, history_lin['train_loss'], 's--', color='#2ecc71', alpha=0.5, label='Linformer (train)')
ax.plot(epochs, history_lin['val_loss'], 's-', color='#2ecc71', linewidth=2, label='Linformer (held-out)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.axhline(np.log(2), color='gray', ls='--', alpha=0.5, label='Random guess (ln 2)')
ax.set_title('Held-Out Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs, history_std['train_acc'], 'o--', color='#e74c3c', alpha=0.5, label='Standard (train)')
ax.plot(epochs, history_std['val_acc'], 'o-', color='#e74c3c', linewidth=2, label='Standard (held-out)')
ax.plot(epochs, history_lin['train_acc'], 's--', color='#2ecc71', alpha=0.5, label='Linformer (train)')
ax.plot(epochs, history_lin['val_acc'], 's-', color='#2ecc71', linewidth=2, label='Linformer (held-out)')
ax.axhline(0.5, color='gray', ls='--', alpha=0.5, label='Random (50%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title(f'Classification Accuracy ({binary_names[0]} vs {binary_names[1]})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal held-out accuracy:")
print(f"  Standard:  {history_std['val_acc'][-1]:.3f}")
print(f"  Linformer: {history_lin['val_acc'][-1]:.3f}")
print(f"\nParameter counts: standard = {sum(p.numel() for p in model_std.parameters()):,}, "
      f"Linformer = {sum(p.numel() for p in model_lin.parameters()):,}")
print("Linformer is slightly more accurate in this run, but the gap is small enough to")
print("treat as run-to-run noise. The tutorial point is not that Linformer wins on jets;")
print("it is that you can cut attention cost without destroying performance.")
print("Notice also that Linformer has more parameters here because the learned sequence")
print("projections E and F add weights. Lower attention complexity does not automatically")
print("mean a smaller model.")

---
## 6. Other Efficient Attention Methods

Linformer is one of several approaches to reducing the quadratic cost:

**Flash Attention** (Dao et al., 2022) computes the exact same attention but uses IO-aware tiling on GPUs to avoid materializing the full $n \times n$ matrix in HBM. Now the default in most frameworks.

**Performer** (Choromanski et al., 2020) approximates the softmax kernel with random features, rearranging the matrix multiplications so the $n \times n$ matrix is never formed. $O(n)$ complexity.

**Sparse Attention** (various) restricts attention to selected pairs (local windows, strided patterns), skipping most entries.

| Method | Time | Memory | Approach |
|---|---|---|---|
| Standard | $O(n^2)$ | $O(n^2)$ | Full attention matrix |
| Flash Attention | $O(n^2)$ | $O(n)$ | IO-aware tiling (exact) |
| Linformer | $O(nk)$ | $O(nk)$ | Key/value projection to lower dim |
| Performer | $O(n)$ | $O(n)$ | Random feature kernel approximation |
| Sparse Attention | $O(n\sqrt{n})$ | $O(n\sqrt{n})$ | Attend to selected pairs only |

For jets with $\sim$128 particles, standard attention is fine. These methods matter more when the object is much larger: full event reconstruction, tracking, or high-granularity calorimeters.

Quick check: why does the Linformer model above have **more** parameters even though its attention is cheaper? Because the cost reduction comes from learned sequence-compression matrices $E$ and $F$, not from removing weights elsewhere.

---
## 7. Summary

- Standard attention costs $O(n^2)$ because it computes all pairwise interactions. We confirmed this empirically -- doubling $n$ makes it 4x slower.

- **Linformer** projects keys and values from $n$ to $k$ with learned projections:
$$\text{Linformer:} \quad \text{softmax}\!\left(\frac{Q(EK)^T}{\sqrt{d_k}}\right)(FV) \quad \text{cost: } O(nk) \text{ instead of } O(n^2)$$

- On the same small QCD-vs-top task from Notebook 1, Linformer achieves comparable accuracy to standard attention.
- Efficient attention changes how computation scales with sequence length. It does **not** automatically reduce parameter count, and it does not guarantee better accuracy on short jet sequences.

- For jets ($n \sim 128$), standard attention is perfectly tractable. Efficient methods become important at larger scales.

**Notebook 3** covers how the Particle Transformer encodes physics directly into the attention mechanism.

### References

- Wang et al., "Linformer: Self-Attention with Linear Complexity" (2020, [arXiv:2006.04768](https://arxiv.org/abs/2006.04768))
- Dao et al., "FlashAttention: Fast and Memory-Efficient Exact Attention" (2022, [arXiv:2205.14135](https://arxiv.org/abs/2205.14135))
- Tay et al., "Efficient Transformers: A Survey" (2020, [arXiv:2009.06732](https://arxiv.org/abs/2009.06732))
- Qu et al., "Particle Transformer for Jet Tagging" (2022, [arXiv:2202.03772](https://arxiv.org/abs/2202.03772))